In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu google-genai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 30.9 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

from google import genai
from google.genai import types

import gradio as gr

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
from google.colab import files

pdf_files = []

while len(pdf_files) < 5:
    print(f"Please upload {5 - len(pdf_files)} more PDF file(s).")

    uploaded = files.upload()

    for file in uploaded.keys():
        if file.lower().endswith(".pdf"):
            pdf_files.append(file)
        else:
            print(f"Skipped: {file} (not a PDF)")

    # Remove duplicates
    pdf_files = list(dict.fromkeys(pdf_files))

print("\nUploaded 5 PDF files:")
for file in pdf_files:
    print("-", file)

Please upload 5 more PDF file(s).


Saving leave_policy (1).pdf to leave_policy (1).pdf
Please upload 4 more PDF file(s).


Saving attendance_policy (1).pdf to attendance_policy (1).pdf
Please upload 3 more PDF file(s).


Saving work_from_home_policy (1).pdf to work_from_home_policy (1).pdf
Please upload 2 more PDF file(s).


Saving reimbursement_policy (1).pdf to reimbursement_policy (1).pdf
Please upload 1 more PDF file(s).


Saving employee_handbook (1).pdf to employee_handbook (1).pdf

Uploaded 5 PDF files:
- leave_policy (1).pdf
- attendance_policy (1).pdf
- work_from_home_policy (1).pdf
- reimbursement_policy (1).pdf
- employee_handbook (1).pdf


In [6]:
documents = []

for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text:
            text = text.strip()

            documents.append({
                "text": text,
                "source": pdf_file,
                "page": page_number
            })

print("Total pages extracted:", len(documents))

Total pages extracted: 10


In [7]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Text cleaning completed!")

Text cleaning completed!


In [8]:
def create_chunks(text, chunk_size=800, overlap=150):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [9]:
def get_policy_type(filename):
    name = filename.lower()

    if "leave" in name:
        return "Leave Policy"
    elif "attendance" in name:
        return "Attendance Policy"
    elif "wfh" in name or "work_from_home" in name or "remote" in name:
        return "Work From Home Policy"
    elif "reimbursement" in name:
        return "Reimbursement Policy"
    elif "handbook" in name:
        return "Employee Handbook"
    else:
        return "General Policy"


chunks = []

for doc in documents:
    text_chunks = create_chunks(doc["text"])

    for chunk in text_chunks:
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "policy_type": get_policy_type(doc["source"])
        })

print("Total chunks:", len(chunks))

Total chunks: 21


In [10]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print("Chunk ID:", i)
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Policy Type:", chunk["policy_type"])
    print("Text:", chunk["text"][:500])

Chunk ID: 0
Source: leave_policy (1).pdf
Page: 1
Policy Type: Leave Policy
Text: Company Leave Policy Sample Policy Document — Fictional Company 1. Purpose This policy explains the types of leave available to employees, the approval process, and the responsibilities of employees when requesting leave. 2. Applicability This policy applies to all full-time and part-time employees of the fictional company unless a separate employment agreement states otherwise. 3. Working Year The leave year runs from 1 January through 31 December. 4. Casual Leave Employees are entitled to 12 c
Chunk ID: 1
Source: leave_policy (1).pdf
Page: 1
Policy Type: Leave Policy
Text: one working day in advance. 5. Sick Leave Employees are entitled to 10 sick leave days per calendar year. If an employee is absent because of illness for more than two consecutive working days, the manager or HR team may request appropriate medical documentation. 6. Earned Leave Employees receive 15 earned leave days per calendar year.

In [11]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [12]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (21, 384)


In [13]:
embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

print("Embeddings normalized!")

Embeddings normalized!


In [14]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS vector database created!")
print("Number of vectors:", index.ntotal)

FAISS vector database created!
Number of vectors: 21


In [15]:
def retrieve_documents(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            result = chunks[idx].copy()
            result["score"] = float(score)
            results.append(result)

    return results

In [16]:
query = "How many casual leaves are allowed?"

results = retrieve_documents(query, top_k=5)

for result in results:
    print("=" * 80)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Policy:", result["policy_type"])
    print("Text:", result["text"][:500])

Score: 0.29331281781196594
Source: leave_policy (1).pdf
Page: 1
Policy: Leave Policy
Text: Company Leave Policy Sample Policy Document — Fictional Company 1. Purpose This policy explains the types of leave available to employees, the approval process, and the responsibilities of employees when requesting leave. 2. Applicability This policy applies to all full-time and part-time employees of the fictional company unless a separate employment agreement states otherwise. 3. Working Year The leave year runs from 1 January through 31 December. 4. Casual Leave Employees are entitled to 12 c
Score: 0.20485320687294006
Source: leave_policy (1).pdf
Page: 1
Policy: Leave Policy
Text: one working day in advance. 5. Sick Leave Employees are entitled to 10 sick leave days per calendar year. If an employee is absent because of illness for more than two consecutive working days, the manager or HR team may request appropriate medical documentation. 6. Earned Leave Employees receive 15 earned leave day

In [17]:
!pip install -q transformers sentencepiece accelerate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully!


In [18]:
SYSTEM_PROMPT = """
You are a Company Policy Assistant.

Your task is to answer employee questions ONLY using
 the
information provided in the retrieved company policy documents.

Rules:

1. Use only the provided context.
2. Do not use outside knowledge.
3. Do not invent or assume policy information.
4. If the answer is not present in the context, say:
   "Information not found in the provided company policy documents."
5. Give a clear and concise answer.
6. Mention the relevant policy when possible.
7. Do not make up document names or page numbers.
"""



In [19]:
def build_context(results):
    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"""
SOURCE {i}
Document: {result['source']}
Page: {result['page']}
Policy Type: {result['policy_type']}

Content:
{result['text']}
"""
        )

    return "\n".join(context_parts)

In [20]:
def rag_answer(query, top_k=3):

    results = retrieve_documents(query, top_k=top_k)

    if not results:
        return (
            "Information not found in the provided company policy documents.",
            []
        )

    best_score = results[0]["score"]

    if best_score < 0.15:
        return (
            "Information not found in the provided company policy documents.",
            []
        )

    context = build_context(results)

    prompt = f"""
You are a company policy question-answering assistant.

Answer the employee's question using ONLY the information in the
provided company policy context.

IMPORTANT:
- Give a complete sentence.
- Give the exact number or rule when available.
- Do not add information that is not present in the context.
- If the answer is not present in the context, say:
Information not found in the provided company policy documents.

Company Policy Context:
{context}

Employee Question:
{query}

Answer in one clear complete sentence:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=50,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, results

In [21]:
question = "How many casual leaves are allowed?"

answer, sources = rag_answer(question)

print("ANSWER:")
print(answer)

print("\nSOURCES:")

for source in sources[:3]:
    print(
        f"- {source['source']} | "
        f"Page {source['page']} | "
        f"{source['policy_type']}"
    )

ANSWER:
12 days per calendar year.

SOURCES:
- leave_policy (1).pdf | Page 1 | Leave Policy
- leave_policy (1).pdf | Page 1 | Leave Policy
- attendance_policy (1).pdf | Page 1 | Attendance Policy


In [22]:
question = "What is the capital of France?"

answer, sources = rag_answer(question)

print(answer)

Information not found in the provided company policy documents.


In [23]:
test_questions = [
    "How many casual leaves are allowed?",
    "What is the work from home policy?",
    "What is the attendance requirement?",
    "What are the reimbursement rules?",
    "How many sick leaves can an employee take?"
]

for question in test_questions:

    print("=" * 80)
    print("QUESTION:", question)

    answer, sources = rag_answer(question)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")

    for source in sources[:2]:
        print(
            f"{source['source']} - Page {source['page']}"
        )

    print()

QUESTION: How many casual leaves are allowed?

ANSWER:
12 days per calendar year.

SOURCES:
leave_policy (1).pdf - Page 1
leave_policy (1).pdf - Page 1

QUESTION: What is the work from home policy?

ANSWER:
Work From Home Policy.

SOURCES:
work_from_home_policy (1).pdf - Page 1
work_from_home_policy (1).pdf - Page 2

QUESTION: What is the attendance requirement?

ANSWER:
90 percent attendance during scheduled working days.

SOURCES:
attendance_policy (1).pdf - Page 1
attendance_policy (1).pdf - Page 1

QUESTION: What are the reimbursement rules?

ANSWER:
Approved claims are normally processed in the next available reimbursement cycle after finance verification.

SOURCES:
reimbursement_policy (1).pdf - Page 1
reimbursement_policy (1).pdf - Page 1

QUESTION: How many sick leaves can an employee take?

ANSWER:
The exact number or rule when available.

SOURCES:
leave_policy (1).pdf - Page 1
leave_policy (1).pdf - Page 1



In [24]:
def format_sources(sources):
    if not sources:
        return "No relevant source found."

    source_text = "\n\n### Sources\n"

    seen = set()

    for source in sources:
        key = (source["source"], source["page"])

        if key not in seen:
            source_text += (
                f"- 📄 **{source['source']}** — "
                f"Page {source['page']} "
                f"({source['policy_type']})\n"
            )

            seen.add(key)

    return source_text

In [25]:
def chatbot(query):

    if not query.strip():
        return "Please enter a question."

    answer, sources = rag_answer(query)

    source_text = format_sources(sources)

    return answer + "\n" + source_text

In [26]:
print(chatbot("What is the work from home policy?"))

Work From Home Policy.


### Sources
- 📄 **work_from_home_policy (1).pdf** — Page 1 (Work From Home Policy)
- 📄 **work_from_home_policy (1).pdf** — Page 2 (Work From Home Policy)
- 📄 **employee_handbook (1).pdf** — Page 1 (Employee Handbook)



In [27]:
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Ask about Company Policies",
        placeholder="Example: How many casual leaves are allowed?"
    ),
    outputs=gr.Markdown(
        label="Policy Assistant"
    ),
    title="Company Policy RAG Assistant",
    description=(
        "Ask questions about company policies. "
        "Answers are generated only from the uploaded policy documents."
    ),
    examples=[
        "How many casual leaves are allowed?",
        "What is the work from home policy?",
        "What is the attendance requirement?",
        "What are the reimbursement rules?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd247113e4a3f7afdc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
